<a href="https://colab.research.google.com/github/kimheeseo/LSCNS/blob/main/convert_tif_to_png_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
from PIL import Image
import os, shutil, csv

drive.mount('/content/drive')

src_dir = "/content/drive/MyDrive/Fiber Category"
dst_dir = "/content/drive/MyDrive/Fiber Category_PNG"

# 1) 대상 파일 수집 (정렬해서 번호가 항상 같은 순서로 매겨지게 함)
items = []
for root, dirs, filenames in os.walk(src_dir):
    dirs.sort()
    for fn in sorted(filenames):
        if fn.lower().endswith((".png", ".tif", ".tiff")):
            items.append(os.path.join(root, fn))

# 2) 이름 변경 + 변환/복사
counters = {}   # (카테고리, HCF여부) → 다음 번호
mapping = []    # 원본 ↔ 새 이름 기록용

for src_path in items:
    rel_parts = os.path.relpath(src_path, src_dir).split(os.sep)
    category = rel_parts[0] if len(rel_parts) > 1 else os.path.basename(src_dir)  # bending / cleaving / fracture
    is_hcf = any(p.lower() == "hcf" for p in rel_parts[1:-1])                    # 하위 경로에 HCF 폴더가 있는지

    key = (category, is_hcf)
    idx = counters.get(key, 0)
    counters[key] = idx + 1

    prefix = f"{category}_HCF" if is_hcf else category
    new_name = f"{prefix}_{idx:03d}.png"

    out_dir = os.path.join(dst_dir, category)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, new_name)

    if src_path.lower().endswith(".png"):
        shutil.copy2(src_path, out_path)          # PNG는 원본 그대로 복사
    else:
        with Image.open(src_path) as im:          # TIF → PNG 변환
            if im.mode not in ("L", "RGB", "RGBA", "P", "I;16"):
                im = im.convert("RGB")
            im.save(out_path)

    mapping.append([os.path.relpath(src_path, src_dir), os.path.join(category, new_name)])

# 3) 원본 ↔ 새 이름 대응표 저장
with open(os.path.join(dst_dir, "rename_mapping.csv"), "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["원본 경로", "새 경로"])
    w.writerows(mapping)

print(f"총 {len(mapping)}개 저장 완료 → {dst_dir}")
for (cat, hcf), n in sorted(counters.items()):
    print(f"  {cat}{'_HCF' if hcf else ''}: {n}개")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
총 583개 저장 완료 → /content/drive/MyDrive/Fiber Category_PNG
  bending: 133개
  cleaving: 274개
  cleaving_HCF: 62개
  fracture: 106개
  fracture_HCF: 8개
